# 📊 Customer Churn Prediction Engine — Exploratory Data Analysis (EDA)

**Objective:**
Identify behavioral patterns, demographic signals, and contract characteristics associated with customer attrition in the IBM Telco Churn dataset.

**Key Questions:**
1. What is the baseline churn rate across our customer population?
2. How does customer tenure relate to attrition risk?
3. Which contract types, internet services, and payment methods show elevated churn rates?
4. How do monthly charges correlate with churn?

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import settings
from src.data_loader import load_raw_data

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
print("Setup completed successfully.")

## 1. Data Ingestion & Quality Inspection

In [ ]:
# Load raw dataset via data_loader
df_raw = load_raw_data()
print(f"Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
# Inspect data types and missing values
df_raw.info()

### Notice: TotalCharges is parsed as string/object
In the IBM Telco dataset, `TotalCharges` contains blank strings (`' '`) for customers with `tenure == 0` (new subscribers who haven't received their first bill). We convert this to numeric, coercing spaces to `NaN`.

In [ ]:
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn_Numeric"] = (df["Churn"] == "Yes").astype(int)

null_total_charges = df["TotalCharges"].isnull().sum()
print(f"TotalCharges blank entries detected & coerced to NaN: {null_total_charges}")

## 2. Target Variable Analysis (Class Imbalance)

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pcts = df["Churn"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(churn_counts.index, churn_counts.values, color=["#2b5c8f", "#d95f02"], width=0.45, edgecolor="black")
for bar in bars:
    h = bar.get_height()
    pct = (h / len(df)) * 100
    ax.annotate(f"{h:,}\n({pct:.1f}%)", xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", fontweight="bold")

ax.set_title("Customer Churn Class Distribution")
ax.set_ylabel("Number of Customers")
ax.set_ylim(0, max(churn_counts.values) * 1.15)
plt.show()

## 3. Churn by Contract Type & Tenure

In [ ]:
contract_churn = df.groupby("Contract")["Churn_Numeric"].mean() * 100
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(contract_churn.index, contract_churn.values, color="#d95f02", edgecolor="black", width=0.45)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f"{h:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", fontweight="bold")

ax.set_title("Churn Rate (%) by Contract Commitment")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, max(contract_churn.values) * 1.2)
plt.show()

In [ ]:
# Tenure Distribution Segmented by Churn
plt.figure(figsize=(9, 4.5))
sns.kdeplot(data=df[df["Churn"] == "No"], x="tenure", label="Retained", fill=True, color="#2b5c8f", alpha=0.3)
sns.kdeplot(data=df[df["Churn"] == "Yes"], x="tenure", label="Churned", fill=True, color="#d95f02", alpha=0.5)
plt.title("Tenure Density by Churn Status (Early-Tenure Cliff Effect)")
plt.xlabel("Tenure (Months)")
plt.ylabel("Density")
plt.legend()
plt.show()

## 4. Churn by Services and Support (Tech Support & Internet Service)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Internet service churn
net_churn = df.groupby("InternetService")["Churn_Numeric"].mean() * 100
ax1.bar(net_churn.index, net_churn.values, color="#7570b3", edgecolor="black", width=0.45)
ax1.set_title("Churn Rate by Internet Service")
ax1.set_ylabel("Churn Rate (%)")
for i, v in enumerate(net_churn.values):
    ax1.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

# Tech support churn
tech_churn = df.groupby("TechSupport")["Churn_Numeric"].mean() * 100
ax2.bar(tech_churn.index, tech_churn.values, color="#1b9e77", edgecolor="black", width=0.45)
ax2.set_title("Churn Rate by Tech Support")
ax2.set_ylabel("Churn Rate (%)")
for i, v in enumerate(tech_churn.values):
    ax2.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Pearson Correlation Matrix

In [ ]:
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Churn_Numeric"]
corr = df[num_cols].corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", vmin=-0.5, vmax=0.5, square=True, mask=mask)
plt.title("Correlation Matrix (Numerical Features & Churn)")
plt.show()